# Notebook 06: Event Correlation and Risk Scoring

Correlate prior communication with a transaction and produce an explainable decision.

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.correlation import correlate_events
from src.features import detect_behaviours
from src.explain import explain_decision

events = pd.read_csv(PROJECT_ROOT / "data" / "synthetic" / "communication_events.csv")
sample_transaction = {
    "transaction_id": "TXN001",
    "user_id": events.iloc[0]["user_id"],
    "timestamp": pd.Timestamp(events.iloc[0]["timestamp"]) + pd.Timedelta(minutes=5),
    "is_new_recipient": True,
}
related = correlate_events(sample_transaction, events, window_minutes=15)
behaviours = sorted({
    behaviour
    for text in related["message"]
    for behaviour in detect_behaviours(text)
})
result = explain_decision(
    sample_transaction["user_id"],
    sample_transaction["transaction_id"],
    {"detected_behaviours": behaviours, "is_new_recipient": True, "new_device": True},
)
display(result)
display(related[["event_id", "event_type", "timestamp", "message"]])

{'user_id': 'USER044',
 'transaction_id': 'TXN001',
 'risk_score': 35,
 'risk_level': 'MEDIUM',
 'recommended_action': 'WARN',
 'detected_behaviours': [],
 'explanation': 'The transfer was directed to an unfamiliar recipient. The account showed a recently registered or changed device.'}

,event_id,event_type,timestamp,message
0,L0119,SMS,2026-01-01 02:03:00+00:00,This is a confirmation that the payment was co...
500,L0119,SMS,2026-01-01 02:03:00+00:00,Your OTP was used to approve your recent accou...
